# Deep Cognitive Agent - Experiment Notebook

This notebook evaluates the **planning quality** of the Deep Cognitive Agent across
three milestones. Unlike the original `test_planning.py` scoring (which reported ~99%
by using metrics that mirrored the prompt template), this notebook applies **rigorous,
independent evaluation** with proper methodology.

## Problems Fixed From Original Evaluation

| Issue | Original Approach | Fixed Approach |
|-------|------------------|----------------|
| Circular scoring | Prompt told LLM to use verbs X; scoring rewarded verbs X | Verb overlap measured independently; dependency-chain analysis added |
| Fake accuracy | Heuristic score called accuracy (no ground truth) | Renamed to Plan Quality Score; no misleading ML terminology |
| No logical-order check | Only checked for duplicate steps | Validates actual dependency ordering via keyword analysis |
| Prompt leakage into scoring | Prompt example = scoring target | Evaluation criteria are independent of prompt wording |
| No edge cases | Only cooperative, well-formed inputs | Added ambiguous, adversarial, and under-specified tasks |
| No cross-milestone comparison | Each milestone tested in isolation | Unified evaluation across M1, M2, M3 |

## Cell 1 - Imports and Environment Setup

In [ ]:
import os
import sys
import json
import re
import time
import statistics
from datetime import datetime
from collections import Counter

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# Disable tracing for experiments to avoid polluting LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "false"

print(f"Project root: {PROJECT_ROOT}")
print(f"GROQ_API_KEY set: {bool(os.getenv('GROQ_API_KEY'))}")
print(f"Timestamp: {datetime.now().isoformat()}")

## Cell 2 - Load Previous Test Results (Offline Analysis)

Before making any API calls, we first analyze the **existing** test outputs
saved in `outputs/`. This lets us audit the original ~99% score.

In [ ]:
outputs_dir = os.path.join(PROJECT_ROOT, "outputs")

# Load the test summary from Milestone 1
summary_path = os.path.join(outputs_dir, "test_summary.json")
with open(summary_path, "r", encoding="utf-8") as f:
    original_summary = json.load(f)

print("=" * 60)
print("ORIGINAL TEST SUMMARY (Milestone 1)")
print("=" * 60)
print(f"Total tests:  {original_summary['total_tests']}")
print(f"Successful:   {original_summary['successful']}")
print(f"Total score:  {original_summary['total_score']}/{original_summary['max_score']}")
print(f'Reported "accuracy": {original_summary["total_score"]/original_summary["max_score"]*100:.1f}%')
print()
print("Per-test breakdown:")
for r in original_summary["results"]:
    s = r["scores"]
    print(f"  Test {r['test_number']}: {s['total']}/20  "
          f"(C={s['clarity']} Co={s['completeness']} Sp={s['specificity']} O={s['logical_order']})  "
          f"- {r['task'][:50]}")

## Cell 3 - Audit: Why the Original Score Was ~99%

We now examine **why** the original scoring produced near-perfect results
by checking for circular dependencies between the prompt instructions and
the evaluation criteria.

In [ ]:
# ---- Load the planning prompt template ----
from tools.planning.write_todos import planning_prompt

prompt_text = planning_prompt.template
print("PLANNING PROMPT (excerpt):")
print(prompt_text[:500])
print("...")

# ---- Load the original scoring verb list from test_planning.py ----
from tests.test_planning import STRONG_VERBS, score_plan

# Extract verbs that the prompt TELLS the LLM to use
prompt_suggested_verbs = [
    "research", "analyze", "summarize", "compare", "propose", "refine"
]

# Check overlap: how many prompt-suggested verbs are in the scoring list?
overlap = [v for v in prompt_suggested_verbs if v in STRONG_VERBS]
overlap_pct = len(overlap) / len(prompt_suggested_verbs) * 100

print(f"\n{'=' * 60}")
print("AUDIT: Prompt-to-Scoring Circularity")
print(f"{'=' * 60}")
print(f"Verbs prompt tells LLM to use: {prompt_suggested_verbs}")
print(f"Verbs scoring rewards:         {len(STRONG_VERBS)} verbs")
print(f"Overlap:                        {len(overlap)}/{len(prompt_suggested_verbs)} ({overlap_pct:.0f}%)")
print(f"Overlapping verbs:              {overlap}")
print()
print("FINDING: The prompt explicitly instructs the LLM to use the exact verbs")
print("that the scoring function rewards. This is circular evaluation --")
print("the test measures prompt-compliance, not genuine planning quality.")
print()

# Check completeness circularity
print("COMPLETENESS CHECK:")
print("  Prompt asks for: '5-8 clear, specific, actionable steps'")
print("  Scoring gives 5/5 for: 4-6 steps")
print("  LLM consistently produces: 6 steps (from test results)")
step_counts = [r["todo_count"] for r in original_summary["results"]]
print(f"  Actual step counts: {step_counts}")
print(f"  All in 4-6 range (auto 5/5): {all(4 <= c <= 6 for c in step_counts)}")
print()

# Check logical order circularity
print("LOGICAL ORDER CHECK:")
print("  Original scoring: only checks for duplicate steps (not actual order)")
print("  Any plan with unique step names gets 5/5 automatically.")
print("  This does NOT validate that steps are in a logical sequence.")

## Cell 4 - Corrected Scoring Function

A rigorous scoring function that evaluates planning quality **independently**
of the prompt template. Key improvements:

- **Dependency chain analysis**: checks if later steps actually reference earlier ones
- **Verb diversity**: rewards variety of action verbs, not just matching a fixed list
- **Logical ordering**: validates research -> comparison -> synthesis -> refinement flow
- **Specificity**: penalizes vague/generic steps rather than just checking verb presence
- **Redundancy detection**: catches semantically similar/overlapping steps

In [ ]:
# ---- Phase keywords for ordering validation ----
PHASE_KEYWORDS = {
    "research": ["research", "investigate", "explore", "study", "examine",
                 "gather", "collect", "identify", "survey"],
    "analysis": ["analyze", "compare", "contrast", "evaluate", "assess",
                 "benchmark", "categorize", "classify"],
    "synthesis": ["propose", "create", "develop", "design", "build",
                  "unify", "synthesize", "integrate", "combine", "formulate"],
    "refinement": ["refine", "improve", "enhance", "optimize", "polish",
                   "revise", "finalize", "validate"]
}

VAGUE_PHRASES = [
    "do the thing", "handle it", "take care of", "work on", "deal with",
    "figure out", "look into", "think about", "consider", "etc",
    "and so on", "various", "stuff", "things"
]


def classify_step_phase(step_text):
    """Classify a step into a workflow phase based on keywords."""
    step_lower = step_text.lower()
    for phase, keywords in PHASE_KEYWORDS.items():
        if any(kw in step_lower for kw in keywords):
            return phase
    return "unknown"


def corrected_score_plan(todos):
    """
    Improved scoring function with independent evaluation criteria.

    Dimensions (1-5 each, max 25):
      1. Completeness  - Appropriate number of steps (4-8)
      2. Specificity    - Steps are concrete, not vague
      3. Verb Diversity - Uses a variety of action verbs (not all the same)
      4. Logical Order  - Steps follow research -> analysis -> synthesis -> refinement
      5. Dependency Chain - Later steps build on earlier steps (keyword overlap)
    """
    tasks = [t["task"] for t in todos]
    n = len(tasks)

    # 1. COMPLETENESS (4-8 steps is good; 5-7 is ideal)
    if 5 <= n <= 7:
        completeness = 5
    elif n == 4 or n == 8:
        completeness = 4
    elif n == 3:
        completeness = 2
    else:
        completeness = 1

    # 2. SPECIFICITY (penalize vague phrases; reward detailed steps)
    vague_count = 0
    too_short = 0
    for t in tasks:
        t_lower = t.lower()
        if any(vp in t_lower for vp in VAGUE_PHRASES):
            vague_count += 1
        if len(t.split()) < 5:
            too_short += 1

    vague_ratio = (vague_count + too_short) / max(n, 1)
    if vague_ratio == 0:
        specificity = 5
    elif vague_ratio <= 0.2:
        specificity = 4
    elif vague_ratio <= 0.4:
        specificity = 3
    elif vague_ratio <= 0.6:
        specificity = 2
    else:
        specificity = 1

    # 3. VERB DIVERSITY (unique first-words as proxy for verb variety)
    first_words = [t.split()[0].lower() if t.split() else "" for t in tasks]
    unique_first = len(set(first_words))
    diversity_ratio = unique_first / max(n, 1)
    if diversity_ratio >= 0.8:
        verb_diversity = 5
    elif diversity_ratio >= 0.6:
        verb_diversity = 4
    elif diversity_ratio >= 0.4:
        verb_diversity = 3
    elif diversity_ratio >= 0.2:
        verb_diversity = 2
    else:
        verb_diversity = 1

    # 4. LOGICAL ORDER (research before synthesis/refinement?)
    phases = [classify_step_phase(t) for t in tasks]
    phase_order = {"research": 0, "analysis": 1, "synthesis": 2,
                   "refinement": 3, "unknown": 1.5}
    order_values = [phase_order[p] for p in phases]

    # Count order violations
    violations = 0
    for i in range(len(order_values) - 1):
        if order_values[i] > order_values[i + 1] + 0.5:
            violations += 1

    if violations == 0:
        logical_order = 5
    elif violations == 1:
        logical_order = 4
    elif violations == 2:
        logical_order = 3
    else:
        logical_order = max(1, 5 - violations)

    # 5. DEPENDENCY CHAIN (later steps reference earlier step concepts?)
    stop_words = {"the", "a", "an", "and", "or", "of", "in", "to", "for",
                  "with", "on", "by", "from", "that", "this", "is", "are",
                  "be", "as", "at", "it"}
    step_words = [set(t.lower().split()) - stop_words for t in tasks]

    chain_links = 0
    for i in range(1, len(step_words)):
        prior_words = set()
        for j in range(i):
            prior_words |= step_words[j]
        shared = step_words[i] & prior_words
        if len(shared) >= 2:
            chain_links += 1

    chain_ratio = chain_links / max(n - 1, 1)
    if chain_ratio >= 0.7:
        dependency_chain = 5
    elif chain_ratio >= 0.5:
        dependency_chain = 4
    elif chain_ratio >= 0.3:
        dependency_chain = 3
    elif chain_ratio >= 0.1:
        dependency_chain = 2
    else:
        dependency_chain = 1

    total = completeness + specificity + verb_diversity + logical_order + dependency_chain
    return {
        "completeness": completeness,
        "specificity": specificity,
        "verb_diversity": verb_diversity,
        "logical_order": logical_order,
        "dependency_chain": dependency_chain,
        "total": total,
        "max": 25,
        "phases_detected": phases,
        "order_violations": violations,
        "chain_links": chain_links
    }


print("Corrected scoring function defined.")
print("Dimensions: Completeness, Specificity, Verb Diversity, Logical Order, Dependency Chain")
print("Max score: 25 (5 dimensions x 5 points)")

## Cell 5 - Re-score Existing Results with Corrected Function

Apply the corrected scoring to the previously saved test outputs. This
reveals the **actual** quality when evaluated fairly.

In [ ]:
# Load individual test result files to get the actual TODO content
test_files = sorted(
    [f for f in os.listdir(outputs_dir)
     if f.startswith("test_") and f.endswith(".json") and "summary" not in f],
    key=lambda x: int(re.search(r"test_(\d+)", x).group(1)) if re.search(r"test_(\d+)", x) else 0
)

print("=" * 70)
print("RE-SCORING WITH CORRECTED EVALUATION")
print("=" * 70)
print(f"{'Test':<6} {'Comp':>4} {'Spec':>4} {'Verb':>4} {'Ordr':>4} {'Deps':>4} {'Total':>6} {'Old':>5}  Task")
print("-" * 90)

corrected_results = []
for tf in test_files:
    filepath = os.path.join(outputs_dir, tf)
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    todos = data.get("todos", [])
    if not todos:
        continue

    new_scores = corrected_score_plan(todos)

    # Find old score from summary
    test_num = int(re.search(r"test_(\d+)", tf).group(1))
    old_score = next(
        (r["scores"]["total"] for r in original_summary["results"]
         if r["test_number"] == test_num), None
    )

    corrected_results.append({
        "test_number": test_num,
        "task": data.get("task", tf),
        "todos": todos,
        "old_score": old_score,
        "new_scores": new_scores
    })

    s = new_scores
    print(f"  {test_num:<4} {s['completeness']:>4} {s['specificity']:>4} {s['verb_diversity']:>4} "
          f"{s['logical_order']:>4} {s['dependency_chain']:>4} {s['total']:>4}/25 "
          f"{old_score or '?':>4}/20  {data.get('task', '')[:40]}")

# Summary comparison
if corrected_results:
    new_total = sum(r["new_scores"]["total"] for r in corrected_results)
    new_max = 25 * len(corrected_results)
    old_total = original_summary["total_score"]
    old_max = original_summary["max_score"]

    print()
    print("=" * 70)
    print(f"Original score:       {old_total}/{old_max}  ({old_total/old_max*100:.1f}%)")
    print(f"Corrected score:      {new_total}/{new_max}  ({new_total/new_max*100:.1f}%)")
    print(f"Score difference:     {old_total/old_max*100 - new_total/new_max*100:+.1f} percentage points")
    print("=" * 70)
    print()
    print("NOTE: A lower score under rigorous evaluation is EXPECTED and HEALTHY.")
    print("The original ~99% was inflated due to circular prompt-to-scoring alignment.")

## Cell 6 - Define Test Inputs (Standard + Edge Cases)

The original test suite used only 8 cooperative, well-structured tasks.
We add **adversarial and edge-case inputs** to properly stress-test the agent.

In [ ]:
# Original standard test inputs
STANDARD_TASKS = [
    "Create a research outline for renewable energy trends",
    "Design a structured learning roadmap for data science",
    "Break down the steps for developing a web application",
    "Plan a comparative study between electric and hydrogen vehicles",
]

# NEW: Edge-case and adversarial inputs
EDGE_CASE_TASKS = [
    # Vague / under-specified
    "Make something cool with AI",
    # Single word
    "Plan",
    # Very long and complex
    ("Design a complete end-to-end machine learning pipeline for predicting "
     "customer churn in a telecom company, including data collection from multiple "
     "sources, feature engineering, model selection, hyperparameter tuning, "
     "deployment to production, A/B testing, monitoring, and continuous retraining"),
    # Non-task input (should the agent still plan?)
    "What is the capital of France?",
]

ALL_TASKS = STANDARD_TASKS + EDGE_CASE_TASKS

print(f"Total test inputs: {len(ALL_TASKS)}")
print(f"  Standard:   {len(STANDARD_TASKS)}")
print(f"  Edge cases: {len(EDGE_CASE_TASKS)}")
for i, t in enumerate(ALL_TASKS, 1):
    label = "STD" if i <= len(STANDARD_TASKS) else "EDGE"
    print(f"  [{label}] {i}. {t[:70]}{'...' if len(t) > 70 else ''}")

## Cell 7 - Run Milestone 1 Evaluation (Live LLM Calls)

This cell makes actual API calls to Groq. It runs the planning agent on
all test inputs and collects results with the **corrected** scoring.

**Note**: This requires a valid `GROQ_API_KEY` in `.env`. Skip this cell
if running offline -- Cell 5 above already re-scored the cached results.

In [ ]:
from app import create_planning_agent, run_agent

agent = create_planning_agent()
print("Planning agent initialized.")
print()

live_results = []
DELAY_BETWEEN_CALLS = 12  # seconds, to respect Groq rate limits

for i, task in enumerate(ALL_TASKS, 1):
    label = "STD" if i <= len(STANDARD_TASKS) else "EDGE"
    print(f"[{label}] Test {i}/{len(ALL_TASKS)}: {task[:60]}...")

    try:
        thread_id = f"experiment-{i}-{datetime.now().strftime('%Y%m%d%H%M%S')}"
        result = run_agent(agent, task, thread_id=thread_id)

        todos = result["todos"]
        if todos:
            scores = corrected_score_plan(todos)
            print(f"  Generated {len(todos)} steps | Score: {scores['total']}/25")
            for j, t in enumerate(todos, 1):
                print(f"    {j}. {t['task']}")
        else:
            scores = {"total": 0, "max": 25, "completeness": 0, "specificity": 0,
                      "verb_diversity": 0, "logical_order": 0, "dependency_chain": 0}
            print("  WARNING: No TODOs generated!")

        live_results.append({
            "test_number": i,
            "category": label,
            "task": task,
            "todos": todos,
            "scores": scores,
            "success": bool(todos)
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        live_results.append({
            "test_number": i,
            "category": label,
            "task": task,
            "todos": [],
            "scores": None,
            "success": False,
            "error": str(e)
        })

    if i < len(ALL_TASKS):
        print(f"  Waiting {DELAY_BETWEEN_CALLS}s for rate limit...")
        time.sleep(DELAY_BETWEEN_CALLS)

print()
print("All tests completed.")

## Cell 8 - Results Summary and Comparison

Compare standard vs edge-case performance and show the corrected overall score.

In [ ]:
print("=" * 75)
print("EXPERIMENT RESULTS SUMMARY")
print("=" * 75)

# Separate standard and edge case results
std_results = [r for r in live_results if r["category"] == "STD" and r["scores"]]
edge_results = [r for r in live_results if r["category"] == "EDGE" and r["scores"]]
failed = [r for r in live_results if not r["success"]]

print(f"{'Category':<15} {'Tests':>5} {'Passed':>6} {'Avg Score':>10} {'Min':>4} {'Max':>4}")
print("-" * 50)

for label, results in [("Standard", std_results), ("Edge Cases", edge_results)]:
    if results:
        scores = [r["scores"]["total"] for r in results]
        print(f"{label:<15} {len(results):>5} {len(results):>6} "
              f"{statistics.mean(scores):>8.1f}/25 {min(scores):>4} {max(scores):>4}")
    else:
        print(f"{label:<15}     0      0         N/A    -    -")

# Overall
all_scored = [r for r in live_results if r["scores"]]
if all_scored:
    all_scores = [r["scores"]["total"] for r in all_scored]
    total_pts = sum(all_scores)
    max_pts = 25 * len(all_scored)

    print("-" * 50)
    print(f"{'OVERALL':<15} {len(all_scored):>5} {len(all_scored):>6} "
          f"{statistics.mean(all_scores):>8.1f}/25 {min(all_scores):>4} {max(all_scores):>4}")
    print(f"\nTotal Plan Quality Score: {total_pts}/{max_pts} ({total_pts/max_pts*100:.1f}%)")

if failed:
    print(f"\nFailed tests ({len(failed)}):")
    for r in failed:
        print(f"  Test {r['test_number']}: {r.get('error', 'No TODOs generated')[:60]}")

print()
print("=" * 75)
print("DIMENSION BREAKDOWN (average across all scored tests)")
print("=" * 75)
if all_scored:
    dims = ["completeness", "specificity", "verb_diversity", "logical_order", "dependency_chain"]
    for dim in dims:
        vals = [r["scores"][dim] for r in all_scored]
        print(f"  {dim:<20} {statistics.mean(vals):>4.1f}/5  (min={min(vals)}, max={max(vals)})")

## Cell 9 - Milestone 2 and 3 Evaluation (From Saved Outputs)

Analyze the Milestone 2 (VFS pipeline) and Milestone 3 (multi-agent)
test results that were previously saved.

In [ ]:
print("=" * 70)
print("MILESTONE 2 & 3 - PIPELINE QUALITY EVALUATION")
print("=" * 70)

for milestone_num, filename in [(2, "milestone2_test_result.json"),
                                 (3, "milestone3_test_result.json")]:
    filepath = os.path.join(outputs_dir, filename)
    if not os.path.exists(filepath):
        print(f"\n  Milestone {milestone_num}: {filename} not found, skipping.")
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        mdata = json.load(f)

    print(f"\n{'=' * 60}")
    print(f"  MILESTONE {milestone_num}")
    print(f"{'=' * 60}")
    print(f"  Task: {mdata.get('task', 'N/A')[:70]}")
    print(f"  Passed: {mdata.get('passed', 'N/A')}")
    print(f"  TODOs: {mdata.get('todo_count', 0)} (done: {mdata.get('todos_done', 0)})")
    print(f"  Step types: {mdata.get('step_types', [])}")
    print(f"  Files in VFS: {mdata.get('files_in_vfs', [])}")
    print(f"  Trace log entries: {mdata.get('trace_log_length', 0)}")

    print(f"  edit_file used: {mdata.get('edit_file_used', False)}")
    print(f"  Selective retrieval: {mdata.get('selective_retrieval', False)}")
    print(f"  Dependency chain valid: {mdata.get('dependency_chain_valid', False)}")
    print(f"  Final output length: {mdata.get('final_output_length', 0)} chars")

    if milestone_num == 3:
        print(f"  Delegation count: {mdata.get('delegation_count', 0)}")
        print(f"  Agents used: {mdata.get('agents_used', [])}")
        print(f"  Delegation reasoning: {mdata.get('delegation_reasoning_count', 0)}")
        print(f"  Result integration valid: {mdata.get('result_integration_valid', False)}")

    errors = mdata.get("errors", [])
    warnings = mdata.get("warnings", [])
    print(f"\n  Errors: {len(errors)}")
    for e in errors:
        print(f"    - {e}")
    print(f"  Warnings: {len(warnings)}")
    for w in warnings:
        print(f"    - {w}")

## Cell 10 - Cross-Milestone Quality Matrix

Unified evaluation table showing which capabilities are validated
across milestones and which checks are meaningful vs. rubber-stamp passes.

In [ ]:
print("=" * 70)
print("CROSS-MILESTONE CAPABILITY MATRIX")
print("=" * 70)
print()

capabilities = [
    ("Dynamic plan generation (LLM-based)",    True,  True,  True),
    ("Structured TODO output",                  True,  True,  True),
    ("Enriched metadata (step_type/deps)",      False, True,  True),
    ("Virtual File System (VFS)",               False, True,  True),
    ("Selective file retrieval",                False, True,  True),
    ("edit_file for refinement",                False, True,  True),
    ("Trace logging",                           False, True,  True),
    ("Multi-agent delegation",                  False, False, True),
    ("Delegation reasoning",                    False, False, True),
    ("Result integration validation",           False, False, True),
    ("Agent attribution in trace",              False, False, True),
]

print(f"  {'Capability':<42} {'M1':>3} {'M2':>3} {'M3':>3}")
print(f"  {'-'*42} {'---':>3} {'---':>3} {'---':>3}")
for cap, m1, m2, m3 in capabilities:
    s1 = 'Yes' if m1 else ' - '
    s2 = 'Yes' if m2 else ' - '
    s3 = 'Yes' if m3 else ' - '
    print(f"  {cap:<42} {s1:>3} {s2:>3} {s3:>3}")

print()
print('KEY FINDING: The M1 "accuracy" of ~99% only measures plan FORMATTING,')
print("not actual planning QUALITY. The corrected evaluation reveals the true")
print("picture -- which is still strong for cooperative inputs but shows real")
print("weaknesses on edge cases and adversarial inputs.")

## Cell 11 - Save Corrected Experiment Results

In [ ]:
experiment_output = {
    "timestamp": datetime.now().isoformat(),
    "description": "Corrected evaluation of Deep Cognitive Agent planning quality",
    "methodology": {
        "scoring_dimensions": ["completeness", "specificity", "verb_diversity",
                               "logical_order", "dependency_chain"],
        "max_score_per_dimension": 5,
        "max_total": 25,
        "improvements_over_original": [
            "Independent verb evaluation (not matching prompt template)",
            "Actual logical-order analysis (not just duplicate detection)",
            "Dependency chain validation via keyword overlap",
            "Edge-case and adversarial test inputs added",
            "Renamed from accuracy to plan quality score"
        ]
    },
    "original_score": {
        "total": original_summary["total_score"],
        "max": original_summary["max_score"],
        "pct": round(original_summary["total_score"] / original_summary["max_score"] * 100, 1)
    },
    "results": []
}

# Add live results if available, otherwise add re-scored cached results
results_to_save = live_results if live_results else []
for r in results_to_save:
    entry = {
        "test_number": r["test_number"],
        "category": r.get("category", "STD"),
        "task": r["task"],
        "todo_count": len(r["todos"]),
        "scores": r["scores"],
        "success": r["success"]
    }
    if "error" in r:
        entry["error"] = r["error"]
    experiment_output["results"].append(entry)

# Compute overall corrected score
scored = [r for r in results_to_save if r.get("scores")]
if scored:
    total = sum(r["scores"]["total"] for r in scored)
    mx = 25 * len(scored)
    experiment_output["corrected_score"] = {
        "total": total,
        "max": mx,
        "pct": round(total / mx * 100, 1)
    }

output_path = os.path.join(outputs_dir, "experiment_corrected_results.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(experiment_output, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {output_path}")
if scored:
    print(f"Corrected Plan Quality Score: {total}/{mx} ({total/mx*100:.1f}%)")

## Cell 12 - Conclusions

### Summary of Findings

1. **The original ~99% accuracy was inflated** due to circular evaluation:
   - The prompt told the LLM exactly which verbs, step counts, and format to use
   - The scoring function rewarded those same verbs, counts, and format
   - The logical order check only detected duplicates, not actual ordering issues
   - The metric was mislabeled as accuracy when it was a format-compliance score

2. **Under rigorous evaluation**, the agent still performs well on standard cooperative
   tasks but shows real weaknesses on edge cases (vague, single-word, or non-task inputs).

3. **The agent does genuinely produce useful plans** for well-specified tasks. The LLM
   generates contextually appropriate steps with reasonable dependency ordering.

4. **Milestones 2 and 3 add genuine architectural capabilities** (VFS, selective retrieval,
   multi-agent delegation) that go beyond simple format compliance.

### Recommendations

- Use Plan Quality Score instead of Accuracy to avoid misleading ML terminology
- Always include edge-case and adversarial inputs in evaluation suites
- Evaluate scoring criteria independently from prompt engineering
- Consider human evaluation or LLM-as-judge for semantic quality assessment